In [1]:
import pandas as pd
import numpy as np


In [2]:
std_summary = pd.read_csv("./UNSUPERVISED/AB/AB_UNS_STD_summary.csv")
std_curve = pd.read_csv("./UNSUPERVISED/AB/AB_UNS_STD_curve_mean.csv")

raw_summary = pd.read_csv("./UNSUPERVISED/AB/AB_UNS_summary.csv")
raw_curve = pd.read_csv("./UNSUPERVISED/AB/AB_UNS_curve_mean.csv")


In [3]:
DELTA_REF = -0.1150
DELTA_TOL = 0.002

In [4]:
def check_curve_order(g, threshold=0.5):
    """
    Sprawdza, czy curve_mean przechodzi tylko w jednym kierunku:

        0 ... 0 -> 1 ... 1

    Dopuszczamy wartości ciągłe mean, więc sprawdzamy
    przejście względem p = 0.5.

    Zwraca:
        valid_order
        n_crossings
    """

    g = g.sort_values("Delta")

    p = g["mean"].to_numpy()

    # usuwamy NaN
    p = p[~np.isnan(p)]

    if len(p) < 2:
        return False, 0

    # wartości po lewej/prawej stronie p=0.5
    below = p < threshold
    above_or_equal = p >= threshold

    # przejścia przez 0.5
    crossings = np.sum(
        below[:-1] & above_or_equal[1:]
    )

    # Jeśli raz przeszliśmy powyżej 0.5,
    # nie możemy już później zejść poniżej 0.5.
    first_above = np.where(above_or_equal)[0]

    if len(first_above) == 0:
        # cała krzywa jest poniżej 0.5
        return False, 0

    first = first_above[0]

    ordered = not np.any(p[first:] < threshold)

    return ordered, crossings


In [5]:
def analyse_variant(summary, curve, variant_name):

    results = []

    group_cols = ["feature_set", "model", "stride"]

    for keys, summary_g in summary.groupby(group_cols):

        feature_set, model, stride = keys

        # ----------------------------------------------------
        # summary
        # ----------------------------------------------------

        # Zakładamy jedną konfigurację w summary
        row = summary_g.iloc[0]

        delta_crit = row["Delta_crit"]
        delta_close = row["Delta_close"]

        # Czy Delta_crit jest zgodne z wartością referencyjną?
        delta_ok = (
            np.isfinite(delta_crit)
            and abs(delta_crit - DELTA_REF) <= DELTA_TOL
        )

        # ----------------------------------------------------
        # curve_mean
        # ----------------------------------------------------

        curve_g = curve[
            (curve["feature_set"] == feature_set)
            & (curve["model"] == model)
            & (curve["stride"] == stride)
        ]

        if len(curve_g) == 0:
            curve_ok = False
            n_crossings = 0
        else:
            curve_ok, n_crossings = check_curve_order(curve_g)

        # ----------------------------------------------------
        # wynik końcowy
        # ----------------------------------------------------

        valid = delta_ok and curve_ok

        results.append({
            "feature_set": feature_set,
            "model": model,
            "stride": stride,

            f"{variant_name}_Delta_crit": delta_crit,
            f"{variant_name}_Delta_close": delta_close,

            f"{variant_name}_delta_ok": delta_ok,
            f"{variant_name}_curve_ok": curve_ok,
            f"{variant_name}_n_crossings": n_crossings,

            f"{variant_name}_valid": valid,
        })

    return pd.DataFrame(results)


In [6]:
std_results = analyse_variant(
    std_summary,
    std_curve,
    "std"
)

raw_results = analyse_variant(
    raw_summary,
    raw_curve,
    "raw"
)


# ============================================================
# POŁĄCZENIE TEJ SAMEJ KONFIGURACJI
# ============================================================

keys = ["feature_set", "model", "stride"]

comparison = pd.merge(
    raw_results,
    std_results,
    on=keys,
    how="outer"
)


# ============================================================
# CZY STANDARYZACJA POPRAWIŁA WYNIK?
# ============================================================

comparison["improvement"] = (
    (~comparison["raw_valid"].fillna(False))
    &
    (comparison["std_valid"].fillna(False))
)


/tmp/ipykernel_22277/2269538636.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (~comparison["raw_valid"].fillna(False))
/tmp/ipykernel_22277/2269538636.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (comparison["std_valid"].fillna(False))


In [7]:
cols = [
    "feature_set",
    "model",
    "stride",

    "raw_Delta_crit",
    "raw_Delta_close",
    "raw_curve_ok",
    "raw_n_crossings",
    "raw_valid",

    "std_Delta_crit",
    "std_Delta_close",
    "std_curve_ok",
    "std_n_crossings",
    "std_valid",

    "improvement",
]

print(comparison[cols].to_string(index=False))

 feature_set           model  stride  raw_Delta_crit  raw_Delta_close raw_curve_ok  raw_n_crossings raw_valid  std_Delta_crit  std_Delta_close std_curve_ok  std_n_crossings std_valid  improvement
          12   Agglomerative      10       -0.127000           -0.128        False              2.0     False             NaN              NaN          NaN              NaN       NaN        False
          12   Agglomerative      50       -0.127000           -0.128        False              2.0     False       -0.127000           -0.128        False              2.0     False        False
          12   Agglomerative     100       -0.127000           -0.128        False              2.0     False             NaN              NaN          NaN              NaN       NaN        False
          12   Agglomerative     250       -0.127000           -0.128        False              2.0     False             NaN              NaN          NaN              NaN       NaN        False
          12   Agglo

In [8]:
improved = comparison[
    comparison["improvement"]
].copy()

print("\n\n=== POPRAWA PO STANDARYZACJI ===")
print(
    improved[
        [
            "feature_set",
            "model",
            "stride",
            "raw_Delta_crit",
            "raw_Delta_close",
            "std_Delta_crit",
            "std_Delta_close",
        ]
    ].to_string(index=False)
)



=== POPRAWA PO STANDARYZACJI ===
 feature_set         model  stride  raw_Delta_crit  raw_Delta_close  std_Delta_crit  std_Delta_close
          12         Birch      50          -0.127           -0.128       -0.115000           -0.128
          20 Agglomerative      50          -0.127           -0.128       -0.115000           -0.128
          20         Birch      50          -0.127           -0.128       -0.115000           -0.128
          20        DBSCAN      10             NaN              NaN       -0.116729           -0.116
          30 Agglomerative      50          -0.127           -0.128       -0.115000           -0.128
          30         Birch      50          -0.127           -0.128       -0.115000           -0.128
